# 04 Build Patient Features

This notebook builds AI-ready patient feature tables from normalized vital signs. It keeps feature engineering separate from the normalized relational import layer.

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DATA_DIR = Path('data/processed')
FEATURES_DATA_DIR = Path('data/features')

PATIENT_FILE = PROCESSED_DATA_DIR / 'patient.csv'
VITAL_SIGNS_FILE = PROCESSED_DATA_DIR / 'vital_signs.csv'

FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
patient_df = pd.read_csv(PATIENT_FILE)
vital_signs_df = pd.read_csv(VITAL_SIGNS_FILE, parse_dates=['measured_at'])

print('Patient shape:', patient_df.shape)
print('Vital signs shape:', vital_signs_df.shape)

Patient shape: (333, 13)
Vital signs shape: (30390, 10)


In [3]:
latest_vitals = (
    vital_signs_df
    .sort_values(['patient_id', 'vital_type', 'measured_at'])
    .groupby(['patient_id', 'vital_type'], as_index=False)
    .tail(1)
)

latest_vitals.head()

,id,patient_id,vital_type,value,unit,measured_at,source_patient_id,source_encounter_id,observation_code,description
90,71ceddd5-a561-45e8-a665-eaaa39667ea8,0171edd4-05e9-4186-bc4a-cec77c49dcd1,BLOOD_PRESSURE_DIASTOLIC,79.0,mmHg,2025-11-19 16:20:11+00:00,4a4e1120-ea05-5be7-9558-b056069c3ad0,4a4e1120-ea05-5be7-5777-3429abffaacd,8462-4,Diastolic Blood Pressure
91,5adf17e7-5217-4c8b-83b0-46f410cc09ec,0171edd4-05e9-4186-bc4a-cec77c49dcd1,BLOOD_PRESSURE_SYSTOLIC,116.0,mmHg,2025-11-19 16:20:11+00:00,4a4e1120-ea05-5be7-9558-b056069c3ad0,4a4e1120-ea05-5be7-5777-3429abffaacd,8480-6,Systolic Blood Pressure
92,c5d51e48-53b7-4c37-8f38-781b62bb562e,0171edd4-05e9-4186-bc4a-cec77c49dcd1,BMI,20.1,kg/m2,2025-11-19 16:20:11+00:00,4a4e1120-ea05-5be7-9558-b056069c3ad0,4a4e1120-ea05-5be7-5777-3429abffaacd,39156-5,Body mass index (BMI) [Ratio]
93,1934885a-0729-4209-ac1a-520e663c0301,0171edd4-05e9-4186-bc4a-cec77c49dcd1,HEART_RATE,99.0,beats/min,2025-11-19 16:20:11+00:00,4a4e1120-ea05-5be7-9558-b056069c3ad0,4a4e1120-ea05-5be7-5777-3429abffaacd,8867-4,Heart rate
94,c96a5767-ceb2-417e-a2c1-703acaef01ba,0171edd4-05e9-4186-bc4a-cec77c49dcd1,HEIGHT,167.9,cm,2025-11-19 16:20:11+00:00,4a4e1120-ea05-5be7-9558-b056069c3ad0,4a4e1120-ea05-5be7-5777-3429abffaacd,8302-2,Body Height


In [4]:
latest_vitals_pivot = latest_vitals.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='first'
).reset_index()

latest_vitals_pivot.columns.name = None
latest_vitals_pivot.head()

,patient_id,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT
0,0171edd4-05e9-4186-bc4a-cec77c49dcd1,79.0,116.0,20.1,NaN,NaN,99.0,167.9,NaN,NaN,56.7
1,0227d060-3c67-46b8-8271-803293b74b0a,86.0,118.0,24.7,NaN,NaN,62.0,170.3,NaN,NaN,71.5
2,025fd5f3-6278-4097-b6cb-2ecbaa471409,87.0,138.0,32.1,NaN,NaN,98.0,170.3,NaN,37.7,93.0
3,02777695-73ea-49fc-9e7a-12082dc00674,102.0,120.0,27.6,212.0,96.8,86.0,166.4,NaN,37.3,76.3
4,03dd92eb-5d8d-4c97-90b1-4dbf349fd89b,75.0,130.0,25.8,NaN,NaN,97.0,179.0,NaN,NaN,82.7


In [5]:
recent_30d = vital_signs_df.copy()
max_ts = recent_30d['measured_at'].max()
cutoff_ts = max_ts - pd.Timedelta(days=30)
recent_30d = recent_30d[recent_30d['measured_at'] >= cutoff_ts].copy()

mean_30d = recent_30d.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='mean'
).reset_index()

mean_30d = mean_30d.add_prefix('avg_30d_')
mean_30d = mean_30d.rename(columns={'avg_30d_patient_id': 'patient_id'})
mean_30d.head()

vital_type,patient_id,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT
0,2234c817-a047-4cf6-86c6-528f1cf00b70,84.0,125.0,27.7,218.7,105.6,84.0,185.2,NaN,NaN,95.0
1,248d7f02-5e68-4219-b620-059a76b41f0b,77.9,148.4,NaN,NaN,NaN,62.0,NaN,100.0,37.1,NaN
2,24f6332b-a918-41c5-b1e2-9497ea6d2206,87.0,122.0,17.1,NaN,NaN,70.0,156.3,NaN,NaN,41.7
3,2e359fc3-1c09-4ce9-8351-f6e0dc1554db,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.3,NaN
4,2ebd70f6-88ed-4ab6-9a23-943826aa7c5f,80.0,132.0,27.7,NaN,72.2,100.0,170.0,NaN,NaN,80.0


In [6]:
weight_rows = vital_signs_df[vital_signs_df['vital_type'] == 'WEIGHT'].copy()
weight_rows = weight_rows.sort_values(['patient_id', 'measured_at'])

weight_trend = weight_rows.groupby('patient_id').agg(
    first_weight=('value', 'first'),
    latest_weight=('value', 'last'),
    first_weight_time=('measured_at', 'first'),
    latest_weight_time=('measured_at', 'last')
).reset_index()

weight_trend['weight_change'] = weight_trend['latest_weight'] - weight_trend['first_weight']
weight_trend.head()

,patient_id,first_weight,latest_weight,first_weight_time,latest_weight_time,weight_change
0,0171edd4-05e9-4186-bc4a-cec77c49dcd1,24.9,56.7,2016-09-28 16:20:11+00:00,2025-11-19 16:20:11+00:00,31.8
1,0227d060-3c67-46b8-8271-803293b74b0a,19.8,71.5,2016-04-27 02:32:11+00:00,2025-06-18 02:32:11+00:00,51.7
2,025fd5f3-6278-4097-b6cb-2ecbaa471409,18.8,93.0,2016-06-20 08:07:15+00:00,2026-02-09 08:07:15+00:00,74.2
3,02777695-73ea-49fc-9e7a-12082dc00674,76.3,76.3,2016-05-12 06:16:21+00:00,2026-03-12 06:16:21+00:00,0.0
4,03dd92eb-5d8d-4c97-90b1-4dbf349fd89b,74.6,82.7,2019-01-06 15:18:59+00:00,2025-01-12 15:18:59+00:00,8.1


In [7]:
patient_features_df = patient_df[['id', 'patient_number', 'birth_date', 'gender']].copy()
patient_features_df = patient_features_df.rename(columns={'id': 'patient_id'})

patient_features_df = patient_features_df.merge(latest_vitals_pivot, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(mean_30d, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(weight_trend[['patient_id', 'weight_change', 'latest_weight_time']], on='patient_id', how='left')

patient_features_df.head()

,patient_id,patient_number,birth_date,gender,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT,weight_change,latest_weight_time
0,07e5d8ea-e03a-4ab1-a338-8900c9a53720,P00000001,2019-09-10,MALE,78.0,113.0,15.8,NaN,NaN,61.0,111.2,NaN,37.8,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4,2025-08-26 08:06:56+00:00
1,7dd641b2-a054-4589-b09e-78338229ed5c,P00000002,2009-08-17,MALE,88.0,128.0,18.9,NaN,NaN,87.0,170.7,NaN,37.5,55.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.8,2025-09-29 02:57:01+00:00
2,8c1fd92d-789f-4068-a250-bb0dfcc06cbc,P00000003,1986-06-01,FEMALE,77.0,132.0,27.6,135.3,NaN,90.0,158.3,NaN,37.2,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2023-08-19 22:33:58+00:00
3,82881aeb-5820-44ba-b3c4-160925a41a20,P00000004,1981-01-17,FEMALE,81.0,97.0,28.2,192.0,82.9,68.0,164.8,NaN,NaN,76.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.7,2025-02-01 19:16:29+00:00
4,9a0c288d-05d3-4cd8-a58d-b8500001dfd9,P00000005,1996-01-06,FEMALE,83.0,123.0,23.1,NaN,NaN,93.0,153.2,NaN,37.4,54.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,2025-03-22 05:51:11+00:00


In [8]:
feature_output_file = FEATURES_DATA_DIR / 'patient_features.csv'
patient_features_df.to_csv(feature_output_file, index=False)

print('Exported:', feature_output_file)

Exported: data\features\patient_features.csv


In [9]:
display(patient_features_df.head(20))

,patient_id,patient_number,birth_date,gender,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT,weight_change,latest_weight_time
0,07e5d8ea-e03a-4ab1-a338-8900c9a53720,P00000001,2019-09-10,MALE,78.0,113.0,15.8,NaN,NaN,61.0,111.2,NaN,37.8,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4,2025-08-26 08:06:56+00:00
1,7dd641b2-a054-4589-b09e-78338229ed5c,P00000002,2009-08-17,MALE,88.0,128.0,18.9,NaN,NaN,87.0,170.7,NaN,37.5,55.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.8,2025-09-29 02:57:01+00:00
2,8c1fd92d-789f-4068-a250-bb0dfcc06cbc,P00000003,1986-06-01,FEMALE,77.0,132.0,27.6,135.3,NaN,90.0,158.3,NaN,37.2,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2023-08-19 22:33:58+00:00
3,82881aeb-5820-44ba-b3c4-160925a41a20,P00000004,1981-01-17,FEMALE,81.0,97.0,28.2,192.0,82.9,68.0,164.8,NaN,NaN,76.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.7,2025-02-01 19:16:29+00:00
4,9a0c288d-05d3-4cd8-a58d-b8500001dfd9,P00000005,1996-01-06,FEMALE,83.0,123.0,23.1,NaN,NaN,93.0,153.2,NaN,37.4,54.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,2025-03-22 05:51:11+00:00
5,1fb58111-b429-4446-a385-407f1f7e3dda,P00000006,2017-10-06,MALE,91.0,120.0,15.8,NaN,NaN,79.0,128.5,NaN,37.6,26.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.2,2025-10-03 01:33:26+00:00
6,4526c60b-7a0f-4923-b39b-5f5b108a86e6,P00000007,2007-04-03,FEMALE,92.0,115.0,22.6,NaN,NaN,67.0,163.1,NaN,NaN,60.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.2,2025-05-27 03:19:07+00:00
7,22f929e2-a69e-4119-a7f0-3dc3eacb33a1,P00000008,1975-05-11,FEMALE,76.0,120.0,30.1,252.2,NaN,97.0,152.6,95.0,37.7,70.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.2,2025-05-11 16:22:47+00:00
8,e6cb98b8-b174-4328-aa89-c409ddd6df2e,P00000009,2023-04-03,FEMALE,81.0,129.0,16.9,NaN,NaN,71.0,96.4,NaN,NaN,15.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.4,2026-03-09 21:56:11+00:00
9,30075c0d-e107-4a54-a55b-4e5262d389fd,P00000010,2010-07-12,MALE,93.0,117.0,16.6,NaN,NaN,96.0,175.2,NaN,NaN,50.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0,2025-08-18 05:59:38+00:00
